This section goes to data exploration.

Run the command 'pip install ipykernel'

In [1]:
import pandas as pd
import os

file_path = "../dataset/original/TotalFeatures-ISCXFlowMeter.csv"

df = pd.read_csv(file_path)

print("Dataset successfully loaded!")
print("Dataset shape:", df.shape)

df.head()

'''
for column in df.columns:
    print(column)
'''

Dataset successfully loaded!
Dataset shape: (631955, 80)


'\nfor column in df.columns:\n    print(column)\n'

In [2]:
print("Available labels:")
print(df["calss"].unique())

print("\nNumber of records per label:")
print(df["calss"].value_counts(dropna=False))

Available labels:
<StringArray>
['benign', 'asware', 'GeneralMalware']
Length: 3, dtype: str

Number of records per label:
calss
benign            471597
asware            155613
GeneralMalware      4745
Name: count, dtype: int64


In [3]:
import numpy as np

# Check missing values
print("Missing values:", df.isnull().sum().sum())

# Check duplicate records
print("Duplicate records:", df.duplicated().sum())

# Check infinite values
numeric_df = df.select_dtypes(include=np.number)

print("Infinite values:",
      np.isinf(numeric_df).sum().sum())

# Check column data types
print("\nColumn data types:")
print(df.dtypes.value_counts())

Missing values: 0
Duplicate records: 268615
Infinite values: 0

Column data types:
int64      55
float64    24
str         1
Name: count, dtype: int64


In [4]:
# Separate network features from classification labels
X = df.drop(columns=["calss"])
y = df["calss"]

# Identify duplicate feature combinations
duplicate_features = X.duplicated()

print("Duplicate feature records:",
      duplicate_features.sum())

# Generate a hash for each feature combination
feature_hash = pd.util.hash_pandas_object(
    X, index=False
)

# Associate each feature combination with its label
duplicate_check = pd.DataFrame({
    "feature_hash": feature_hash,
    "label": y
})

# Find feature combinations associated with multiple labels
label_counts = duplicate_check.groupby(
    "feature_hash"
)["label"].nunique()

conflicting_groups = label_counts[label_counts > 1]

print("Feature combinations with conflicting labels:",
      len(conflicting_groups))

print("Records belonging to conflicting groups:",
      duplicate_check["feature_hash"].isin(
          conflicting_groups.index
      ).sum())

Duplicate feature records: 270325
Feature combinations with conflicting labels: 1645
Records belonging to conflicting groups: 74483


In [5]:
# Get feature combinations associated with conflicting labels
conflicting_hashes = conflicting_groups.index

# Retrieve all records belonging to those groups
conflicting_records = duplicate_check[
    duplicate_check["feature_hash"].isin(conflicting_hashes)
]

# Display the number of records associated with each label
print("Labels involved in conflicting records:")
print(conflicting_records["label"].value_counts())

# Display some examples of conflicting feature combinations
print("\nExamples of conflicting records:")

example_hashes = conflicting_hashes[:5]

for h in example_hashes:
    print("\nFeature combination:", h)

    print(
        duplicate_check[
            duplicate_check["feature_hash"] == h
        ]["label"].value_counts()
    )

Labels involved in conflicting records:
label
benign            48584
asware            25290
GeneralMalware      609
Name: count, dtype: int64

Examples of conflicting records:

Feature combination: 8017174940102772
label
asware    3
benign    1
Name: count, dtype: int64

Feature combination: 15753557758780856
label
asware    1
benign    1
Name: count, dtype: int64

Feature combination: 18003471816364911
label
GeneralMalware    1
benign            1
Name: count, dtype: int64

Feature combination: 18928384327380343
label
benign    1
asware    1
Name: count, dtype: int64

Feature combination: 21993168356294717
label
asware    4
benign    1
Name: count, dtype: int64


In [6]:
# Identify feature combinations with conflicting labels
conflicting_hashes = conflicting_groups.index

# Remove all records belonging to conflicting combinations
clean_df = df[
    ~feature_hash.isin(conflicting_hashes)
].copy()

print("Original dataset size:", len(df))
print("Records removed:", len(df) - len(clean_df))
print("Remaining records:", len(clean_df))

print("\nRemaining class distribution:")
print(clean_df["calss"].value_counts())

print("\nRemaining class percentages:")
print(
    clean_df["calss"].value_counts(normalize=True) * 100
)

Original dataset size: 631955
Records removed: 74483
Remaining records: 557472

Remaining class distribution:
calss
benign            423013
asware            130323
GeneralMalware      4136
Name: count, dtype: int64

Remaining class percentages:
calss
benign            75.880582
asware            23.377497
GeneralMalware     0.741921
Name: proportion, dtype: float64


In [7]:
# Separate features and labels from the cleaned dataset
clean_X = clean_df.drop(columns=["calss"])

# Count repeated feature combinations
remaining_duplicates = clean_X.duplicated().sum()

# Count unique feature combinations
unique_combinations = len(clean_X.drop_duplicates())

print("Remaining duplicate feature records:", remaining_duplicates)
print("Unique feature combinations:", unique_combinations)

# Check the class distribution after removing repeated feature combinations
deduplicated_df = clean_df.drop_duplicates(
    subset=clean_X.columns
)

print("\nClass distribution after deduplication:")
print(deduplicated_df["calss"].value_counts())

print("\nClass percentages after deduplication:")
print(
    deduplicated_df["calss"].value_counts(normalize=True) * 100
)

Remaining duplicate feature records: 197487
Unique feature combinations: 359985

Class distribution after deduplication:
calss
benign            236369
asware            120071
GeneralMalware      3545
Name: count, dtype: int64

Class percentages after deduplication:
calss
benign            65.660791
asware            33.354445
GeneralMalware     0.984763
Name: proportion, dtype: float64


In [8]:
# Create a copy of the deduplicated dataset
model_df = deduplicated_df.copy()

# Rename the label column
model_df.rename(columns={"calss": "class"}, inplace=True)

# Correct the Adware label
model_df["class"] = model_df["class"].replace({
    "asware": "Adware",
    "benign": "Benign",
    "GeneralMalware": "General Malware"
})

# Verify the changes
print("Dataset shape:", model_df.shape)

print("\nUpdated column names:")
print(model_df.columns[-5:].tolist())

print("\nUpdated classification labels:")
print(model_df["class"].value_counts())

print("\nMissing values:", model_df.isnull().sum().sum())

Dataset shape: (359985, 80)

Updated column names:
['Init_Win_bytes_backward', 'RRT_samples_clnt', 'Act_data_pkt_forward', 'min_seg_size_forward', 'class']

Updated classification labels:
class
Benign             236369
Adware             120071
General Malware      3545
Name: count, dtype: int64

Missing values: 0


In [9]:
# Generate descriptive statistics for all numerical features
feature_summary = model_df.drop(
    columns=["class"]
).describe().T

# Display selected statistical information
print(
    feature_summary[
        ["mean", "std", "min", "25%", "50%", "75%", "max"]
    ].head(20)
)

                        mean           std   min      25%       50%  \
duration        3.832125e+07  2.505606e+08 -18.0  45357.0  481252.0   
total_fpackets  1.098320e+01  2.306645e+02   0.0      1.0       2.0   
total_bpackets  1.820519e+01  4.628193e+02   0.0      0.0       1.0   
total_fpktl     1.483292e+03  1.091075e+05   0.0     71.0     164.0   
total_bpktl     2.115949e+04  6.391027e+05   0.0      0.0     116.0   
min_fpktl       6.315396e+01  6.889647e+01  -1.0     52.0      52.0   
min_bpktl       6.702840e+01  1.000717e+02  -1.0     -1.0      52.0   
max_fpktl       2.766218e+02  3.499015e+02  -1.0     52.0      77.0   
max_bpktl       3.103452e+02  4.499114e+02  -1.0     -1.0      64.0   
mean_fpktl      1.214019e+02  1.222067e+02   0.0     52.0      67.5   
mean_bpktl      1.735318e+02  2.480030e+02   0.0      0.0      60.0   
std_fpktl       9.226734e+01  1.511362e+02   0.0      0.0       1.0   
std_bpktl       1.075302e+02  1.955001e+02   0.0      0.0       0.0   
total_

In [10]:
# Select numerical features only
X = model_df.drop(columns=["class"])

# Count negative values in each feature
negative_counts = (X < 0).sum()

# Keep only features containing negative values
negative_counts = negative_counts[negative_counts > 0]

# Display results
print("Features containing negative values:")
print(negative_counts.sort_values(ascending=False))

print("\nTotal negative values across the dataset:")
print((X < 0).sum().sum())

Features containing negative values:
min_biat                   209190
max_biat                   209190
min_active                 204559
min_idle                   204559
max_idle                   204559
max_active                 204559
max_fiat                   101195
min_fiat                   101195
Init_Win_bytes_backward     98938
min_bpktl                   98938
max_bpktl                   98938
bVarianceDataBytes           4277
min_fpktl                    3769
Init_Win_bytes_forward       3769
max_fpktl                    3769
min_flowiat                  3315
max_flowiat                  2096
duration                        1
mean_flowiat                    1
dtype: int64

Total negative values across the dataset:
1756817


In [11]:
# Examine the most common negative values in each affected feature
for column in negative_counts.index:

    negative_values = X.loc[X[column] < 0, column]

    print(f"\n{column}")
    print(negative_values.value_counts().head(10))


duration
duration
-18    1
Name: count, dtype: int64

min_fpktl
min_fpktl
-1    3769
Name: count, dtype: int64

min_bpktl
min_bpktl
-1    98938
Name: count, dtype: int64

max_fpktl
max_fpktl
-1    3769
Name: count, dtype: int64

max_bpktl
max_bpktl
-1    98938
Name: count, dtype: int64

min_fiat
min_fiat
-1    101195
Name: count, dtype: int64

min_biat
min_biat
-1    209190
Name: count, dtype: int64

max_fiat
max_fiat
-1    101195
Name: count, dtype: int64

max_biat
max_biat
-1    209190
Name: count, dtype: int64

min_flowiat
min_flowiat
-1     2130
-7       58
-8       48
-18      43
-14      41
-20      39
-6       39
-21      39
-9       38
-19      37
Name: count, dtype: int64

max_flowiat
max_flowiat
-1     2095
-18       1
Name: count, dtype: int64

mean_flowiat
mean_flowiat
-18.0    1
Name: count, dtype: int64

bVarianceDataBytes
bVarianceDataBytes
-0.200000    1837
-1.000000    1244
-0.166667     639
-0.142857     227
-0.500000     160
-0.250000      65
-0.333333      53
-0.12

In [12]:
# Create a compact summary of negative values
negative_summary = []

for column in negative_counts.index:

    values = X.loc[X[column] < 0, column]

    negative_summary.append({
        "Feature": column,
        "Negative Count": len(values),
        "Negative Values": sorted(values.unique().tolist())
    })

negative_summary = pd.DataFrame(negative_summary)

display(negative_summary)

# Investigate the unusual negative duration
print("\nRecord containing negative duration:")

display(
    model_df.loc[
        model_df["duration"] < 0,
        ["duration", "class"]
    ]
)

,Feature,Negative Count,Negative Values
0,duration,1,[-18]
1,min_fpktl,3769,[-1]
2,min_bpktl,98938,[-1]
3,max_fpktl,3769,[-1]
4,max_bpktl,98938,[-1]
5,min_fiat,101195,[-1]
6,min_biat,209190,[-1]
7,max_fiat,101195,[-1]
8,max_biat,209190,[-1]
9,min_flowiat,3315,"[-181, -74, -63, -62, -60, -59, -58, -57, -56,..."



Record containing negative duration:


,duration,class
584673,-18,Benign


In [13]:
# Investigate unusual negative values
features_to_check = [
    "min_flowiat",
    "max_flowiat",
    "mean_flowiat",
    "bVarianceDataBytes"
]

for column in features_to_check:
    print(f"\n--- {column} ---")

    negative_values = model_df.loc[
        model_df[column] < 0, column
    ]

    print("Number of negative records:", len(negative_values))
    print("Most common negative values:")
    print(negative_values.value_counts().head(15))

    print("Minimum value:", model_df[column].min())
    print("Maximum value:", model_df[column].max())

# Examine the record with negative duration
print("\n--- Negative duration record ---")

display(
    model_df.loc[model_df["duration"] < 0].T
)


--- min_flowiat ---
Number of negative records: 3315
Most common negative values:
min_flowiat
-1     2130
-7       58
-8       48
-18      43
-14      41
-20      39
-6       39
-21      39
-9       38
-19      37
-16      35
-15      33
-29      32
-12      32
-10      31
Name: count, dtype: int64
Minimum value: -181
Maximum value: 26647710229

--- max_flowiat ---
Number of negative records: 2096
Most common negative values:
max_flowiat
-1     2095
-18       1
Name: count, dtype: int64
Minimum value: -18
Maximum value: 44310722867

--- mean_flowiat ---
Number of negative records: 1
Most common negative values:
mean_flowiat
-18.0    1
Name: count, dtype: int64
Minimum value: -18.0
Maximum value: 26600000000.0

--- bVarianceDataBytes ---
Number of negative records: 4277
Most common negative values:
bVarianceDataBytes
-0.200000    1837
-1.000000    1244
-0.166667     639
-0.142857     227
-0.500000     160
-0.250000      65
-0.333333      53
-0.125000      20
-0.111111       5
-0.066667

,584673
duration,-18
total_fpackets,1
total_bpackets,1
total_fpktl,1252
total_bpktl,40
...,...
Init_Win_bytes_backward,0
RRT_samples_clnt,0
Act_data_pkt_forward,1
min_seg_size_forward,20


In [14]:
features_to_check = [
    "min_flowiat",
    "max_flowiat",
    "mean_flowiat",
    "bVarianceDataBytes"
]

for column in features_to_check:
    print(f"\n{'=' * 50}")
    print(f"FEATURE: {column}")
    print(f"{'=' * 50}")

    negative_values = model_df.loc[
        model_df[column] < 0, column
    ]

    print("Negative records:", len(negative_values))
    print("Unique negative values:", negative_values.nunique())
    print("Negative value distribution:")
    print(
        negative_values.value_counts().sort_index().to_string()
    )


FEATURE: min_flowiat
Negative records: 3315
Unique negative values: 63
Negative value distribution:
min_flowiat
-181       1
-74        1
-63        4
-62        1
-60        2
-59        1
-58        1
-57        1
-56        4
-55        1
-54        2
-53        1
-52        1
-51        1
-49        3
-48        3
-47        4
-46        1
-45        6
-44        4
-43        3
-42        5
-41        6
-40        8
-39       13
-38       14
-37        9
-36       14
-35       18
-34       25
-33       20
-32       27
-31       19
-30       30
-29       32
-28       20
-27       30
-26       25
-25       30
-24       31
-23       31
-22       26
-21       39
-20       39
-19       37
-18       43
-17       28
-16       35
-15       33
-14       41
-13       28
-12       32
-11       31
-10       31
-9        38
-8        48
-7        58
-6        39
-5        27
-4        26
-3        29
-2        24
-1      2130

FEATURE: max_flowiat
Negative records: 2096
Unique negative values:

In [15]:
features_to_check = [
    "min_flowiat",
    "max_flowiat",
    "mean_flowiat",
    "bVarianceDataBytes"
]

negative_analysis = []

for column in features_to_check:

    negative_values = model_df.loc[
        model_df[column] < 0, column
    ]

    counts = negative_values.value_counts().sort_index()

    for value, count in counts.items():
        negative_analysis.append({
            "Feature": column,
            "Negative_Value": value,
            "Frequency": count
        })

negative_analysis_df = pd.DataFrame(negative_analysis)

negative_analysis_df.to_csv(
    "negative_value_analysis.csv",
    index=False
)

print("Analysis exported successfully!")
print("Total negative-value combinations:", len(negative_analysis_df))
print(negative_analysis_df.groupby("Feature").size())

Analysis exported successfully!
Total negative-value combinations: 93
Feature
bVarianceDataBytes    27
max_flowiat            2
mean_flowiat           1
min_flowiat           63
dtype: int64


In [16]:
# Display complete negative-value distributions
for column in [
    "max_flowiat",
    "mean_flowiat",
    "bVarianceDataBytes"
]:
    print(f"\n--- {column} ---")

    print(
        negative_analysis_df[
            negative_analysis_df["Feature"] == column
        ].to_string(index=False)
    )

# Display min_flowiat range
print("\n--- min_flowiat range ---")
print("Minimum:", X["min_flowiat"].min())
print("Maximum:", X["min_flowiat"].max())


--- max_flowiat ---
    Feature  Negative_Value  Frequency
max_flowiat           -18.0          1
max_flowiat            -1.0       2095

--- mean_flowiat ---
     Feature  Negative_Value  Frequency
mean_flowiat           -18.0          1

--- bVarianceDataBytes ---
           Feature  Negative_Value  Frequency
bVarianceDataBytes       -1.000000       1244
bVarianceDataBytes       -0.500000        160
bVarianceDataBytes       -0.333333         53
bVarianceDataBytes       -0.250000         65
bVarianceDataBytes       -0.200000       1837
bVarianceDataBytes       -0.166667        639
bVarianceDataBytes       -0.142857        227
bVarianceDataBytes       -0.125000         20
bVarianceDataBytes       -0.111111          5
bVarianceDataBytes       -0.100000          1
bVarianceDataBytes       -0.090909          1
bVarianceDataBytes       -0.083333          3
bVarianceDataBytes       -0.076923          1
bVarianceDataBytes       -0.071429          1
bVarianceDataBytes       -0.066667        

In [17]:
# Check whether -1 corresponds to unavailable packet measurements

checks = {
    "min_fpktl": (
        model_df["min_fpktl"] == -1,
        model_df["total_fpackets"] == 0
    ),
    "max_fpktl": (
        model_df["max_fpktl"] == -1,
        model_df["total_fpackets"] == 0
    ),
    "min_bpktl": (
        model_df["min_bpktl"] == -1,
        model_df["total_bpackets"] == 0
    ),
    "max_bpktl": (
        model_df["max_bpktl"] == -1,
        model_df["total_bpackets"] == 0
    ),
    "min_fiat": (
        model_df["min_fiat"] == -1,
        model_df["total_fpackets"] < 2
    ),
    "min_biat": (
        model_df["min_biat"] == -1,
        model_df["total_bpackets"] < 2
    ),
    "Init_Win_bytes_forward": (
        model_df["Init_Win_bytes_forward"] == -1,
        model_df["total_fpackets"] == 0
    ),
    "Init_Win_bytes_backward": (
        model_df["Init_Win_bytes_backward"] == -1,
        model_df["total_bpackets"] == 0
    )
}

for feature, (negative_mask, condition) in checks.items():

    total = negative_mask.sum()
    matching = (negative_mask & condition).sum()

    percentage = (
        matching / total * 100 if total > 0 else 0
    )

    print(f"\n{feature}")
    print("Records containing -1:", total)
    print("Records matching expected condition:", matching)
    print("Percentage matching:", round(percentage, 2), "%")


min_fpktl
Records containing -1: 3769
Records matching expected condition: 3769
Percentage matching: 100.0 %

max_fpktl
Records containing -1: 3769
Records matching expected condition: 3769
Percentage matching: 100.0 %

min_bpktl
Records containing -1: 98938
Records matching expected condition: 98938
Percentage matching: 100.0 %

max_bpktl
Records containing -1: 98938
Records matching expected condition: 98938
Percentage matching: 100.0 %

min_fiat
Records containing -1: 101195
Records matching expected condition: 101195
Percentage matching: 100.0 %

min_biat
Records containing -1: 209190
Records matching expected condition: 209190
Percentage matching: 100.0 %

Init_Win_bytes_forward
Records containing -1: 3769
Records matching expected condition: 3769
Percentage matching: 100.0 %

Init_Win_bytes_backward
Records containing -1: 98938
Records matching expected condition: 98938
Percentage matching: 100.0 %


In [18]:
# Summarize the sentinel-value checks
check_results = []

for feature, (negative_mask, condition) in checks.items():

    total = negative_mask.sum()
    matching = (negative_mask & condition).sum()

    check_results.append({
        "Feature": feature,
        "Records with -1": total,
        "Matching condition": matching,
        "Non-matching records": total - matching,
        "Match percentage": round(
            matching / total * 100, 2
        ) if total > 0 else 0
    })

check_summary = pd.DataFrame(check_results)

display(check_summary)

,Feature,Records with -1,Matching condition,Non-matching records,Match percentage
0,min_fpktl,3769,3769,0,100.0
1,max_fpktl,3769,3769,0,100.0
2,min_bpktl,98938,98938,0,100.0
3,max_bpktl,98938,98938,0,100.0
4,min_fiat,101195,101195,0,100.0
5,min_biat,209190,209190,0,100.0
6,Init_Win_bytes_forward,3769,3769,0,100.0
7,Init_Win_bytes_backward,98938,98938,0,100.0


In [19]:
# Investigate active and idle feature values

active_idle_features = [
    "min_active",
    "max_active",
    "min_idle",
    "max_idle"
]

# Check negative-value counts
print("Negative-value counts:")
print((model_df[active_idle_features] < 0).sum())

# Check whether the -1 values occur in the same records
print("\nAre the -1 positions identical?")

for feature in active_idle_features:
    print(
        feature,
        (model_df[feature] == -1).equals(
            model_df["min_active"] == -1
        )
    )

# Examine records where all four features equal -1
all_unavailable = (
    model_df[active_idle_features] == -1
).all(axis=1)

print("\nRecords where all four features equal -1:")
print(all_unavailable.sum())

# Examine their flow durations
print("\nDuration statistics for these records:")
print(model_df.loc[all_unavailable, "duration"].describe())

# Compare with records where these measurements are available
print("\nDuration statistics for other records:")
print(model_df.loc[~all_unavailable, "duration"].describe())

Negative-value counts:
min_active    204559
max_active    204559
min_idle      204559
max_idle      204559
dtype: int64

Are the -1 positions identical?
min_active True
max_active True
min_idle True
max_idle True

Records where all four features equal -1:
204559

Duration statistics for these records:
count    2.045590e+05
mean     1.954056e+05
std      8.285882e+05
min     -1.800000e+01
25%      7.995000e+03
50%      6.787100e+04
75%      2.393770e+05
max      5.999982e+07
Name: duration, dtype: float64

Duration statistics for other records:
count    1.554260e+05
mean     8.849936e+07
std      3.754677e+08
min      1.000001e+06
25%      7.691494e+06
50%      3.080038e+07
75%      5.998529e+07
max      4.431076e+10
Name: duration, dtype: float64


In [20]:
# Investigate negative bVarianceDataBytes values

negative_variance = model_df["bVarianceDataBytes"] < 0

print("Records with negative values:")
print(negative_variance.sum())

print("\nClass distribution among records with negative values:")
print(model_df.loc[negative_variance, "class"].value_counts())

print("\nClass distribution among records with non-negative values:")
print(model_df.loc[~negative_variance, "class"].value_counts())

print("\nStatistics for negative values:")
print(model_df.loc[negative_variance, "bVarianceDataBytes"].describe())

print("\nStatistics for non-negative values:")
print(model_df.loc[~negative_variance, "bVarianceDataBytes"].describe())

Records with negative values:
4277

Class distribution among records with negative values:
class
Benign             3676
Adware              582
General Malware      19
Name: count, dtype: int64

Class distribution among records with non-negative values:
class
Benign             232693
Adware             119489
General Malware      3526
Name: count, dtype: int64

Statistics for negative values:
count    4277.000000
mean       -0.436901
std         0.366343
min        -1.000000
25%        -1.000000
50%        -0.200000
75%        -0.200000
max        -0.000033
Name: bVarianceDataBytes, dtype: float64

Statistics for non-negative values:
count    355708.000000
mean      50381.532536
std      112604.854863
min           0.000000
25%           0.000000
50%           0.000000
75%       17424.666670
max      895121.000000
Name: bVarianceDataBytes, dtype: float64


In [21]:
# Examine negative bVarianceDataBytes values by class

negative_variance = model_df[
    model_df["bVarianceDataBytes"] < 0
]

print("Negative values by class:")
print(
    negative_variance.groupby("class")["bVarianceDataBytes"]
    .agg(["count", "min", "max", "mean", "median"])
)

print("\nMost common negative values:")
print(
    negative_variance["bVarianceDataBytes"]
    .value_counts()
    .head(15)
)

Negative values by class:
                 count  min       max      mean  median
class                                                  
Adware             582 -1.0 -0.020408 -0.798516    -1.0
Benign            3676 -1.0 -0.000033 -0.377817    -0.2
General Malware     19 -1.0 -0.000158 -0.791236    -1.0

Most common negative values:
bVarianceDataBytes
-0.200000    1837
-1.000000    1244
-0.166667     639
-0.142857     227
-0.500000     160
-0.250000      65
-0.333333      53
-0.125000      20
-0.111111       5
-0.066667       4
-0.020833       3
-0.083333       3
-0.050000       2
-0.052632       2
-0.071429       1
Name: count, dtype: int64


In [22]:
# Count negative values in the remaining features

remaining_features = [
    "min_flowiat",
    "max_flowiat",
    "mean_flowiat",
    "duration"
]

for feature in remaining_features:
    negative = model_df[model_df[feature] < 0]

    print(f"\n{feature}")
    print("Negative records:", len(negative))
    print("Minimum:", negative[feature].min())
    print("Maximum:", negative[feature].max())
    print("Class distribution:")
    print(negative["class"].value_counts())


min_flowiat
Negative records: 3315
Minimum: -181
Maximum: -1
Class distribution:
class
Benign             2012
Adware             1244
General Malware      59
Name: count, dtype: int64

max_flowiat
Negative records: 2096
Minimum: -18
Maximum: -1
Class distribution:
class
Benign             1436
Adware              624
General Malware      36
Name: count, dtype: int64

mean_flowiat
Negative records: 1
Minimum: -18.0
Maximum: -18.0
Class distribution:
class
Benign    1
Name: count, dtype: int64

duration
Negative records: 1
Minimum: -18
Maximum: -18
Class distribution:
class
Benign    1
Name: count, dtype: int64


In [23]:
# Investigate negative flow inter-arrival times

features = ["min_flowiat", "max_flowiat", "mean_flowiat"]

for feature in features:

    negative = model_df[feature] < 0

    print(f"\n--- {feature} ---")
    print("Negative records:", negative.sum())

    print("\nForward packet count:")
    print(model_df.loc[negative, "total_fpackets"].describe())

    print("\nBackward packet count:")
    print(model_df.loc[negative, "total_bpackets"].describe())

    print("\nMost common negative values:")
    print(model_df.loc[negative, feature].value_counts().head(10))


--- min_flowiat ---
Negative records: 3315

Forward packet count:
count     3315.000000
mean       607.768326
std       2315.196476
min          0.000000
25%          1.000000
50%          1.000000
75%        366.000000
max      48255.000000
Name: total_fpackets, dtype: float64

Backward packet count:
count     3315.000000
mean      1346.488688
std       4604.614976
min          0.000000
25%          0.000000
50%          0.000000
75%        807.500000
max      74768.000000
Name: total_bpackets, dtype: float64

Most common negative values:
min_flowiat
-1     2130
-7       58
-8       48
-18      43
-14      41
-20      39
-6       39
-21      39
-9       38
-19      37
Name: count, dtype: int64

--- max_flowiat ---
Negative records: 2096

Forward packet count:
count    2096.000000
mean        0.968034
std         0.175951
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: total_fpackets, dtype: float64

Backward packet count:

In [24]:
for feature in ["min_flowiat", "max_flowiat"]:

    print(f"\n--- {feature} ---")

    print(
        model_df.loc[
            model_df[feature] < 0,
            feature
        ].value_counts().sort_index().head(30)
    )

    print("\nNumber of distinct negative values:")
    print(
        model_df.loc[
            model_df[feature] < 0,
            feature
        ].nunique()
    )


--- min_flowiat ---
min_flowiat
-181     1
-74      1
-63      4
-62      1
-60      2
-59      1
-58      1
-57      1
-56      4
-55      1
-54      2
-53      1
-52      1
-51      1
-49      3
-48      3
-47      4
-46      1
-45      6
-44      4
-43      3
-42      5
-41      6
-40      8
-39     13
-38     14
-37      9
-36     14
-35     18
-34     25
Name: count, dtype: int64

Number of distinct negative values:
63

--- max_flowiat ---
max_flowiat
-18       1
-1     2095
Name: count, dtype: int64

Number of distinct negative values:
2


In [25]:
# Compare negative minimum and maximum flow inter-arrival times

min_negative = model_df["min_flowiat"] < 0
max_negative = model_df["max_flowiat"] < 0

print("Both minimum and maximum are negative:",
      (min_negative & max_negative).sum())

print("Only minimum is negative:",
      (min_negative & ~max_negative).sum())

print("Only maximum is negative:",
      (~min_negative & max_negative).sum())

print("Neither is negative:",
      (~min_negative & ~max_negative).sum())

Both minimum and maximum are negative: 2096
Only minimum is negative: 1219
Only maximum is negative: 0
Neither is negative: 356670


In [26]:
# Inspect the record with negative duration

negative_duration = model_df[
    model_df["duration"] < 0
]

print("Number of negative duration records:",
      len(negative_duration))

negative_duration.T

Number of negative duration records: 1


,584673
duration,-18
total_fpackets,1
total_bpackets,1
total_fpktl,1252
total_bpktl,40
...,...
Init_Win_bytes_backward,0
RRT_samples_clnt,0
Act_data_pkt_forward,1
min_seg_size_forward,20


In [27]:
# Check whether negative flow IAT values coincide
# with negative duration values

negative_flowiat = (
    (model_df["min_flowiat"] < 0) |
    (model_df["max_flowiat"] < 0)
)

negative_duration = model_df["duration"] < 0

print("Negative flow IAT records:", negative_flowiat.sum())

print("Negative flow IAT records with negative duration:",
      (negative_flowiat & negative_duration).sum())

print("Negative flow IAT records with non-negative duration:",
      (negative_flowiat & ~negative_duration).sum())

Negative flow IAT records: 3315
Negative flow IAT records with negative duration: 1
Negative flow IAT records with non-negative duration: 3314


In [28]:
# Create an experimental dataset

preprocessed_df = model_df[
    model_df["duration"] >= 0
].copy()

print("Original dataset:", model_df.shape)
print("Experimental dataset:", preprocessed_df.shape)

print("\nClass distribution:")
print(preprocessed_df["class"].value_counts())

print("\nNegative duration records remaining:",
      (preprocessed_df["duration"] < 0).sum())

Original dataset: (359985, 80)
Experimental dataset: (359984, 80)

Class distribution:
class
Benign             236368
Adware             120071
General Malware      3545
Name: count, dtype: int64

Negative duration records remaining: 0


In [29]:
import os

# Create the processed dataset directory
os.makedirs("../dataset/processed", exist_ok=True)

# Save the experimental dataset
preprocessed_df.to_csv(
    "../dataset/processed/adware_experimental.csv",
    index=False
)

print("Experimental dataset saved successfully!")
print("Location: dataset/processed/adware_experimental.csv")

Experimental dataset saved successfully!
Location: dataset/processed/adware_experimental.csv


In [30]:
# Investigate negative bVarianceDataBytes records

negative_variance = preprocessed_df[
    preprocessed_df["bVarianceDataBytes"] < 0
]

print("Negative bVarianceDataBytes records:",
      len(negative_variance))

print("\nBackward packet count:")
print(negative_variance["total_bpackets"].describe())

print("\nBackward packet bytes:")
print(negative_variance["total_bpktl"].describe())

print("\nRecords with zero backward packets:",
      (negative_variance["total_bpackets"] == 0).sum())

print("Records with one backward packet:",
      (negative_variance["total_bpackets"] == 1).sum())

print("Records with more than one backward packet:",
      (negative_variance["total_bpackets"] > 1).sum())

Negative bVarianceDataBytes records: 4277

Backward packet count:
count     4277.000000
mean        13.642740
std        474.653795
min          2.000000
25%          2.000000
50%          6.000000
75%          6.000000
max      30393.000000
Name: total_bpackets, dtype: float64

Backward packet bytes:
count    4.277000e+03
mean     1.298811e+04
std      6.598725e+05
min      8.000000e+01
25%      1.280000e+02
50%      1.278000e+03
75%      1.278000e+03
max      4.224627e+07
Name: total_bpktl, dtype: float64

Records with zero backward packets: 0
Records with one backward packet: 0
Records with more than one backward packet: 4277


In [31]:
# Investigate negative min_flowiat records

negative_iat = preprocessed_df[
    preprocessed_df["min_flowiat"] < 0
]

print("Negative min_flowiat records:", len(negative_iat))

print("\nForward packet counts:")
print(negative_iat["total_fpackets"].describe())

print("\nBackward packet counts:")
print(negative_iat["total_bpackets"].describe())

print("\nPacket count combinations:")
print(
    negative_iat[
        ["total_fpackets", "total_bpackets"]
    ].value_counts().head(15)
)

Negative min_flowiat records: 3314

Forward packet counts:
count     3314.000000
mean       607.951418
std       2315.521857
min          0.000000
25%          1.000000
50%          1.000000
75%        366.500000
max      48255.000000
Name: total_fpackets, dtype: float64

Backward packet counts:
count     3314.000000
mean      1346.894689
std       4605.250510
min          0.000000
25%          0.000000
50%          0.000000
75%        808.250000
max      74768.000000
Name: total_bpackets, dtype: float64

Packet count combinations:
total_fpackets  total_bpackets
1               0                 2028
0               1                   67
821             2121                 3
709             2122                 3
163             395                  3
585             2121                 2
740             2122                 2
643             2121                 2
741             2121                 2
729             2123                 2
758             2122                 2
78

In [32]:
# Identify negative min_flowiat records
# with at least two packets in both directions

multiple_packets = (
    (negative_iat["total_fpackets"] >= 2) &
    (negative_iat["total_bpackets"] >= 2)
)

print("Negative min_flowiat records:", len(negative_iat))

print("Records with at least 2 packets in both directions:",
      multiple_packets.sum())

print("\nPercentage:")
print(f"{multiple_packets.mean() * 100:.2f}%")

print("\nExamples:")
print(
    negative_iat.loc[
        multiple_packets,
        [
            "duration",
            "total_fpackets",
            "total_bpackets",
            "min_flowiat",
            "max_flowiat",
            "mean_flowiat",
            "class"
        ]
    ].head(10)
)

Negative min_flowiat records: 3314
Records with at least 2 packets in both directions: 1219

Percentage:
36.78%

Examples:
       duration  total_fpackets  total_bpackets  min_flowiat  max_flowiat  \
0       1020586             668            1641          -30        20674   
2350   67512287              67             109          -25     60036538   
3292  262669175             710            2143          -16    259341675   
3298  265545550             622            1740          -34    260043927   
3300  265545544             976            2688          -20    259920921   
3302  265545548             767            2117          -15    259936794   
3304    6225755            1168            2941          -20      3584993   
4038   48914512              46              49           -7     16711503   
6018    1511867             280             340           -1       482757   
7329  155271526            3761           12223          -31    147819900   

       mean_flowiat   class  

In [33]:
# Compare class distributions for negative and non-negative min_flowiat

negative_iat = preprocessed_df["min_flowiat"] < 0

print("Negative min_flowiat records:")
print(
    preprocessed_df.loc[negative_iat, "class"].value_counts()
)

print("\nNon-negative min_flowiat records:")
print(
    preprocessed_df.loc[~negative_iat, "class"].value_counts()
)

print("\nPercentage of each class with negative min_flowiat:")

for class_name in preprocessed_df["class"].unique():

    class_records = preprocessed_df[
        preprocessed_df["class"] == class_name
    ]

    percentage = (
        (class_records["min_flowiat"] < 0).sum()
        / len(class_records)
    ) * 100

    print(f"{class_name}: {percentage:.2f}%")

Negative min_flowiat records:
class
Benign             2011
Adware             1244
General Malware      59
Name: count, dtype: int64

Non-negative min_flowiat records:
class
Benign             234357
Adware             118827
General Malware      3486
Name: count, dtype: int64

Percentage of each class with negative min_flowiat:
Benign: 0.85%
Adware: 1.04%
General Malware: 1.66%


In [34]:
class_counts = preprocessed_df["class"].value_counts()

print("Largest class:", class_counts.max())
print("Smallest class:", class_counts.min())

print(
    "Imbalance ratio:",
    round(class_counts.max() / class_counts.min(), 2)
)

Largest class: 236368
Smallest class: 3545
Imbalance ratio: 66.68


In [35]:
# Calculate feature correlations

correlation_matrix = preprocessed_df.drop(
    columns=["class"]
).corr()

# Extract pairs with correlation above 0.95
import numpy as np

upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

high_correlations = (
    upper_triangle.stack()
    .reset_index()
)

high_correlations.columns = [
    "Feature 1",
    "Feature 2",
    "Correlation"
]

high_correlations = high_correlations[
    high_correlations["Correlation"].abs() > 0.95
]

high_correlations = high_correlations.sort_values(
    by="Correlation",
    key=abs,
    ascending=False
)

print("Highly correlated feature pairs:")
print(high_correlations.to_string(index=False))

print("\nTotal highly correlated pairs:",
      len(high_correlations))

Highly correlated feature pairs:
       Feature 1            Feature 2  Correlation
     total_fhlen         fHeaderBytes     1.000000
   mean_flowpktl        avgPacketSize     1.000000
      mean_fpktl      fAvgSegmentSize     1.000000
  total_bpackets          total_bhlen     1.000000
     max_flowiat             max_idle     1.000000
      mean_bpktl      bAvgSegmentSize     0.999999
     mean_active            mean_idle     0.999998
      min_active             min_idle     0.999998
      max_active             max_idle     0.999998
     max_flowiat           max_active     0.999998
        min_fiat          min_flowiat     0.999912
  total_fpackets Act_data_pkt_forward     0.999866
  total_fpackets          total_fhlen     0.999838
  total_fpackets         fHeaderBytes     0.999838
      std_active             std_idle     0.999831
      min_active            mean_idle     0.999745
      min_active          mean_active     0.999744
        min_idle            mean_idle     0.99974

In [36]:
# Check the number of unique values in each feature

feature_columns = preprocessed_df.drop(
    columns=["class"]
)

unique_counts = feature_columns.nunique().sort_values()

print("Features with the fewest unique values:")
print(unique_counts.head(20))

# Identify constant features
constant_features = unique_counts[
    unique_counts == 1
]

print("\nConstant features:")
print(constant_features)

print("\nNumber of constant features:",
      len(constant_features))

Features with the fewest unique values:
furg_cnt                  1
burg_cnt                  1
flow_urg                  1
flow_cwr                  1
flow_ece                  1
flow_rst                  3
flow_fin                  3
min_seg_size_forward      5
flow_syn                 10
fAvgPacketsPerBulk       58
FFNEPD                   59
bAvgPacketsPerBulk       59
fpsh_cnt                133
min_flowpktl            182
bpsh_cnt                274
flow_psh                310
min_fpktl               448
min_bpktl               641
sflow_fpacket           658
sflow_bpacket           759
dtype: int64

Constant features:
furg_cnt    1
burg_cnt    1
flow_urg    1
flow_cwr    1
flow_ece    1
dtype: int64

Number of constant features: 5


In [37]:
# Save feature investigation results

os.makedirs("../results", exist_ok=True)

# Save constant features
constant_features.to_csv(
    "../results/constant_features.csv",
    header=["unique_values"]
)

# Save highly correlated feature pairs
high_correlations.to_csv(
    "../results/high_correlations.csv",
    index=False
)

print("Exploration results saved successfully!")

Exploration results saved successfully!
